
# 03 — Tanore Q1 Two-Stream Boro Rice Classification

## Experiments

### Stream A — Fused-Hybrid
- 25 January: fused 4-band image + fused NDVI
- 6 March: fused 4-band image + fused NDVI
- 7 April: fused 4-band image + fused NDVI
- 22 April: prepared Planet 4-band image + Planet NDVI

### Stream B — Planet-only
- All four dates: prepared Planet 4-band image + Planet NDVI

## Models
1. Data-calibrated phenology rule-based classifier  
2. Random Forest  
3. XGBoost  

## Q1-oriented outputs
- Spatially independent validation or independent validation layer
- Spatial group cross-validation for tuning
- Training-only decision-threshold optimization
- Cluster-bootstrap 95% confidence intervals
- OA, balanced accuracy, precision, recall, specificity, F1, Kappa, MCC
- ROC-AUC, PR-AUC and Brier score
- Raw and normalized confusion matrices
- ROC and precision-recall curves
- RF permutation importance
- XGBoost gain importance and SHAP contribution
- Rule threshold table
- Classification, probability and uncertainty GeoTIFFs
- 300-DPI classification and probability maps
- Stream disagreement maps
- Consensus maps
- Mapped-area table
- McNemar paired tests and paired bootstrap differences
- Publication-ready CSV, Excel, PNG and JSON files

## Required reference samples

Preferred Q1 design:

- `Tanore_training_samples.gpkg`
- `Tanore_validation_samples.gpkg`

Both layers must contain a binary class field:

- `1` = Boro rice
- `0` = Non-Boro

Point or polygon samples are supported. A single combined sample layer is also supported as a weaker fallback; the notebook then makes a spatially grouped holdout split.


## Your current sample-folder structure is supported

The notebook automatically reads and merges these four folders:

- `Data/Tanore/Samples/Tanore_rice_training` → Boro = 1
- `Data/Tanore/Samples/Tanore_NonRice_Training` → Non-Boro = 0
- `Data/Tanore/Samples/Tanore_Rice_Validation` → Boro = 1
- `Data/Tanore/Samples/Tanore_NonRice_Validation` → Non-Boro = 0

No class field is required inside the shapefiles because the class is assigned from the folder name.

## VS Code local-PC version

এই notebook Google Colab বা local PC ব্যবহার করে না।

মূল project path:

```text
D:\Boro Rice Classification
```

Notebook চালানোর আগে `setup_windows.bat` চালিয়ে `Python (Boro Rice Project)` kernel নির্বাচন করুন।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:
# CELL 1 — Portable project path for local use and GitHub reproduction

from pathlib import Path
import os

# Recommended: set BORO_PROJECT_ROOT to the local project directory.
# If it is not set, launch Jupyter from the repository root.
PROJECT_ROOT = Path(
    os.environ.get("BORO_PROJECT_ROOT", str(Path.cwd()))
).expanduser().resolve()

DATA_ROOT = PROJECT_ROOT / "Data"
OUTPUTS_ROOT = PROJECT_ROOT / "Outputs"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Data directory was not found: {DATA_ROOT}\n"
        "Set BORO_PROJECT_ROOT or launch Jupyter from the repository root."
    )

OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Outputs root:", OUTPUTS_ROOT)


In [ ]:
# CELL 2 — Import installed local packages

from contextlib import ExitStack
from pathlib import Path
import json
import math
import platform
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.windows import Window, from_bounds
from rasterio.transform import xy
from shapely import make_valid

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from scipy.stats import binomtest
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_val_predict,
)
from xgboost import XGBClassifier, DMatrix
import xgboost as xgb
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

print("Python:", platform.python_version())
print("Rasterio:", rasterio.__version__)
print("GeoPandas:", gpd.__version__)
print("XGBoost:", xgb.__version__)

In [ ]:
# CELL 3 — Local project paths and Q1 settings

STUDY_AREA = "Tanore"

PREPARED_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Prepared_Planet"
)

PLANET_NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Planet_NDVI"
)

FUSION_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion"
)

FUSION_NDVI_DIR = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Fusion_NDVI"
)

SAMPLE_ROOT = (
    DATA_ROOT
    / STUDY_AREA
    / "Samples"
)

OUTPUT_ROOT = (
    OUTPUTS_ROOT
    / STUDY_AREA
    / "Classification_Q1"
)

TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
MAP_DIR = OUTPUT_ROOT / "maps"
MODEL_DIR = OUTPUT_ROOT / "models"
REPORT_DIR = OUTPUT_ROOT / "reports"

for folder in [
    OUTPUT_ROOT,
    TABLE_DIR,
    FIGURE_DIR,
    MAP_DIR,
    MODEL_DIR,
    REPORT_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# Automatic four-folder reading is used.
TRAIN_VECTOR = None
VALIDATION_VECTOR = None
COMBINED_VECTOR = None
CLASS_FIELD = None

POSITIVE_VALUES = {
    "1",
    "rice",
    "boro",
    "boro rice",
    "boro_rice",
    "paddy",
}

NEGATIVE_VALUES = {
    "0",
    "nonrice",
    "non-rice",
    "non rice",
    "non_boro",
    "non-boro",
    "other",
}

RANDOM_SEED = 42
MAX_PIXELS_PER_FEATURE = 300
POINT_BUFFER_METERS = 6.0
SPATIAL_BLOCK_SIZE_METERS = 1000.0
HOLDOUT_FRACTION = 0.25

CV_SPLITS = 5
N_BOOTSTRAP = 1000

MAP_BLOCK_SIZE = 512
FLOAT_NODATA = -9999.0
CLASS_NODATA = 255

QUICK_MODE = False

if QUICK_MODE:
    RF_ITERATIONS = 5
    XGB_ITERATIONS = 5
    N_BOOTSTRAP = 200
else:
    RF_ITERATIONS = 15
    XGB_ITERATIONS = 20

RF_SEARCH_ITERATIONS = RF_ITERATIONS
XGB_SEARCH_ITERATIONS = XGB_ITERATIONS
BOOTSTRAP_REPLICATES = N_BOOTSTRAP
print("Prepared input:", PREPARED_DIR)
print("Fusion input:", FUSION_DIR)
print("Corrected samples:", SAMPLE_ROOT)
print("Classification output:", OUTPUT_ROOT)
print("Example expected input:", PREPARED_DIR / "Tanore_25_jan_SR_prepared_clip.tif")


In [ ]:
# CELL 4 — Define and verify the two data streams

DATES = ["Jan", "Mar", "Apr1", "Apr2"]

STREAMS = {
    "FusedHybrid": {
        "bands": {
            "Jan": FUSION_DIR / "Fused_Tanore_25_jan.tif",
            "Mar": FUSION_DIR / "Fused_Tanore_6_march.tif",
            "Apr1": FUSION_DIR / "Fused_Tanore_7_april.tif",
            "Apr2": PREPARED_DIR / "Tanore_22_april_SR_prepared_clip.tif",
        },
        "ndvi": {
            "Jan": FUSION_NDVI_DIR / "NDVI_Fused_Tanore_25_jan.tif",
            "Mar": FUSION_NDVI_DIR / "NDVI_Fused_Tanore_6_march.tif",
            "Apr1": FUSION_NDVI_DIR / "NDVI_Fused_Tanore_7_april.tif",
            "Apr2": PLANET_NDVI_DIR / "Tanore_22_april_NDVI_clip.tif",
        },
    },
    "PlanetOnly": {
        "bands": {
            "Jan": PREPARED_DIR / "Tanore_25_jan_SR_prepared_clip.tif",
            "Mar": PREPARED_DIR / "Tanore_6_march_SR_prepared_clip.tif",
            "Apr1": PREPARED_DIR / "Tanore_7_april_SR_prepared_clip.tif",
            "Apr2": PREPARED_DIR / "Tanore_22_april_SR_prepared_clip.tif",
        },
        "ndvi": {
            "Jan": PLANET_NDVI_DIR / "Tanore_25_jan_NDVI_clip.tif",
            "Mar": PLANET_NDVI_DIR / "Tanore_6_march_NDVI_clip.tif",
            "Apr1": PLANET_NDVI_DIR / "Tanore_7_april_NDVI_clip.tif",
            "Apr2": PLANET_NDVI_DIR / "Tanore_22_april_NDVI_clip.tif",
        },
    },
}

inventory_rows = []
reference_grid = None
reference_path = None

for stream_name, stream in STREAMS.items():
    for data_type in ["bands", "ndvi"]:
        expected_count = 4 if data_type == "bands" else 1

        for date_name, path in stream[data_type].items():
            if not path.exists():
                raise FileNotFoundError(
                    f"Required input is missing: {path}"
                )

            with rasterio.open(path) as src:
                if src.count != expected_count:
                    raise ValueError(
                        f"{path.name}: expected {expected_count} band(s), "
                        f"found {src.count}."
                    )

                grid = (
                    str(src.crs),
                    src.transform,
                    src.width,
                    src.height,
                )

                if reference_grid is None:
                    reference_grid = grid
                    reference_path = path
                elif grid != reference_grid:
                    raise ValueError(
                        f"Grid mismatch: {path}. "
                        "All classification inputs must share one grid."
                    )

                inventory_rows.append({
                    "stream": stream_name,
                    "type": data_type,
                    "date": date_name,
                    "path": str(path),
                    "bands": src.count,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "resolution_x": abs(float(src.transform.a)),
                    "resolution_y": abs(float(src.transform.e)),
                    "nodata": src.nodata,
                })

input_inventory = pd.DataFrame(inventory_rows)
display(input_inventory)

input_inventory.to_csv(
    TABLE_DIR / "Q1_Input_Data_Inventory.csv",
    index=False,
)

print("✅ All two-stream inputs exist and share the same grid.")
print("Reference raster:", reference_path)


In [ ]:
# CELL 5 — Feature definitions and raster helper functions

BAND_NAMES = ["Blue", "Green", "Red", "NIR"]

BASE_FEATURE_NAMES = []

for date_name in DATES:
    for band_name in BAND_NAMES:
        BASE_FEATURE_NAMES.append(
            f"{date_name}_{band_name}"
        )

    BASE_FEATURE_NAMES.append(
        f"{date_name}_NDVI"
    )

DERIVED_FEATURE_NAMES = [
    "NDVI_mean",
    "NDVI_std",
    "NDVI_min",
    "NDVI_max",
    "NDVI_amplitude",
    "dNDVI_Mar_Jan",
    "dNDVI_Apr1_Mar",
    "dNDVI_Apr2_Apr1",
    "dNDVI_Apr1_Jan",
    "NDVI_peak_timing",
]

FEATURE_NAMES = BASE_FEATURE_NAMES + DERIVED_FEATURE_NAMES

RULE_FEATURES = [
    "NDVI_max",
    "NDVI_amplitude",
    "dNDVI_Apr1_Jan",
    "Mar_NDVI",
]


def iter_windows(width, height, block_size=512):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off,
                row_off,
                min(block_size, width - col_off),
                min(block_size, height - row_off),
            )


def clamp_window(window, width, height):
    col0 = max(0, int(math.floor(window.col_off)))
    row0 = max(0, int(math.floor(window.row_off)))

    col1 = min(
        width,
        int(math.ceil(window.col_off + window.width)),
    )

    row1 = min(
        height,
        int(math.ceil(window.row_off + window.height)),
    )

    if col1 <= col0 or row1 <= row0:
        return None

    return Window(
        col0,
        row0,
        col1 - col0,
        row1 - row0,
    )


def raster_valid(data, nodata):
    valid = np.all(np.isfinite(data), axis=0)

    if nodata is not None:
        valid &= np.all(data != nodata, axis=0)

    return valid


def compute_feature_cube(
    band_arrays,
    ndvi_arrays,
    band_nodata,
    ndvi_nodata,
):
    feature_layers = []
    valid = None
    ndvi_stack = []

    for date_name in DATES:
        bands = band_arrays[date_name].astype("float32")
        ndvi = ndvi_arrays[date_name].astype("float32")

        current_valid = (
            raster_valid(
                bands,
                band_nodata[date_name],
            )
            & raster_valid(
                ndvi[np.newaxis, :, :],
                ndvi_nodata[date_name],
            )
        )

        valid = (
            current_valid.copy()
            if valid is None
            else valid & current_valid
        )

        for band_index in range(4):
            feature_layers.append(
                bands[band_index]
            )

        feature_layers.append(ndvi)
        ndvi_stack.append(ndvi)

    ndvi_stack = np.stack(
        ndvi_stack,
        axis=0,
    )

    feature_layers.extend([
        np.mean(ndvi_stack, axis=0),
        np.std(ndvi_stack, axis=0),
        np.min(ndvi_stack, axis=0),
        np.max(ndvi_stack, axis=0),
        np.max(ndvi_stack, axis=0)
        - np.min(ndvi_stack, axis=0),
        ndvi_stack[1] - ndvi_stack[0],
        ndvi_stack[2] - ndvi_stack[1],
        ndvi_stack[3] - ndvi_stack[2],
        ndvi_stack[2] - ndvi_stack[0],
        np.argmax(ndvi_stack, axis=0).astype(
            "float32"
        ),
    ])

    cube = np.stack(
        feature_layers,
        axis=0,
    ).astype("float32")

    valid &= np.all(
        np.isfinite(cube),
        axis=0,
    )

    return cube, valid


def open_stream_handles(stack, stream_name):
    stream = STREAMS[stream_name]

    handles = {
        "bands": {},
        "ndvi": {},
    }

    for date_name in DATES:
        handles["bands"][date_name] = stack.enter_context(
            rasterio.open(
                stream["bands"][date_name]
            )
        )

        handles["ndvi"][date_name] = stack.enter_context(
            rasterio.open(
                stream["ndvi"][date_name]
            )
        )

    return handles


def read_feature_window(handles, window):
    band_arrays = {}
    ndvi_arrays = {}
    band_nodata = {}
    ndvi_nodata = {}

    for date_name in DATES:
        band_src = handles["bands"][date_name]
        ndvi_src = handles["ndvi"][date_name]

        band_arrays[date_name] = band_src.read(
            window=window,
        ).astype("float32")

        ndvi_arrays[date_name] = ndvi_src.read(
            1,
            window=window,
        ).astype("float32")

        band_nodata[date_name] = (
            band_src.nodata
            if band_src.nodata is not None
            else FLOAT_NODATA
        )

        ndvi_nodata[date_name] = (
            ndvi_src.nodata
            if ndvi_src.nodata is not None
            else FLOAT_NODATA
        )

    return compute_feature_cube(
        band_arrays,
        ndvi_arrays,
        band_nodata,
        ndvi_nodata,
    )


print("Total predictors:", len(FEATURE_NAMES))
print(FEATURE_NAMES)


In [ ]:
# CELL 6 — Read four Tanore class-specific shapefile folders

VECTOR_SUFFIXES = {
    ".shp",
    ".gpkg",
    ".geojson",
    ".json",
}


def normalize_text(value):
    return (
        str(value)
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_class_folder(
    role,
    is_nonrice,
):
    candidates = []

    if not SAMPLE_ROOT.exists():
        raise FileNotFoundError(
            f"Sample folder not found: {SAMPLE_ROOT}"
        )

    for folder in SAMPLE_ROOT.iterdir():
        if not folder.is_dir():
            continue

        normalized = normalize_text(
            folder.name
        )

        has_tanore = (
            "tanore" in normalized
        )

        has_role = (
            role in normalized
        )

        has_nonrice = (
            "nonrice" in normalized
        )

        has_rice = (
            "rice" in normalized
        )

        class_match = (
            has_nonrice
            if is_nonrice
            else (
                has_rice
                and not has_nonrice
            )
        )

        if (
            has_tanore
            and has_role
            and class_match
        ):
            candidates.append(folder)

    if len(candidates) != 1:
        class_name = (
            "NonRice"
            if is_nonrice
            else "Rice"
        )

        raise FileNotFoundError(
            f"Could not uniquely find Tanore {class_name} "
            f"{role} folder. Matches="
            f"{[str(path) for path in candidates]}"
        )

    return candidates[0]


def choose_vector_file(folder):
    candidates = sorted(
        path
        for path in folder.iterdir()
        if (
            path.is_file()
            and path.suffix.lower()
            in VECTOR_SUFFIXES
        )
    )

    if not candidates:
        raise FileNotFoundError(
            f"No .shp/.gpkg/.geojson found in: {folder}"
        )

    shapefiles = [
        path
        for path in candidates
        if path.suffix.lower() == ".shp"
    ]

    if len(shapefiles) == 1:
        return shapefiles[0]

    if len(shapefiles) > 1:
        return max(
            shapefiles,
            key=lambda path: path.stat().st_size,
        )

    if len(candidates) == 1:
        return candidates[0]

    return max(
        candidates,
        key=lambda path: path.stat().st_size,
    )


with rasterio.open(
    reference_path
) as reference:
    REFERENCE_CRS = reference.crs
    REFERENCE_TRANSFORM = (
        reference.transform
    )
    REFERENCE_WIDTH = (
        reference.width
    )
    REFERENCE_HEIGHT = (
        reference.height
    )
    REFERENCE_BOUNDS = (
        reference.bounds
    )

if REFERENCE_CRS is None:
    raise ValueError(
        "Reference raster CRS is missing."
    )

if REFERENCE_CRS.is_geographic:
    raise ValueError(
        "The reference raster must use a projected CRS."
    )


def read_class_specific_layer(
    folder,
    class_value,
    role,
):
    vector_path = choose_vector_file(
        folder
    )

    gdf = gpd.read_file(
        vector_path
    )

    if gdf.empty:
        raise ValueError(
            f"Empty sample layer: {vector_path}"
        )

    if gdf.crs is None:
        raise ValueError(
            f"CRS is missing: {vector_path}"
        )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
    ].copy()

    gdf["geometry"] = (
        gdf.geometry.apply(
            make_valid
        )
    )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
    ].copy()

    if gdf.empty:
        raise ValueError(
            f"No valid geometries: {vector_path}"
        )

    gdf = gdf.to_crs(
        REFERENCE_CRS
    )

    gdf["_class"] = int(
        class_value
    )

    gdf["_source_role"] = role
    gdf["_source_file"] = str(
        vector_path
    )

    return gdf


rice_training_folder = (
    find_class_folder(
        role="training",
        is_nonrice=False,
    )
)

nonrice_training_folder = (
    find_class_folder(
        role="training",
        is_nonrice=True,
    )
)

rice_validation_folder = (
    find_class_folder(
        role="validation",
        is_nonrice=False,
    )
)

nonrice_validation_folder = (
    find_class_folder(
        role="validation",
        is_nonrice=True,
    )
)


rice_training = (
    read_class_specific_layer(
        rice_training_folder,
        class_value=1,
        role="train",
    )
)

nonrice_training = (
    read_class_specific_layer(
        nonrice_training_folder,
        class_value=0,
        role="train",
    )
)

rice_validation = (
    read_class_specific_layer(
        rice_validation_folder,
        class_value=1,
        role="validation",
    )
)

nonrice_validation = (
    read_class_specific_layer(
        nonrice_validation_folder,
        class_value=0,
        role="validation",
    )
)


training_layer = gpd.GeoDataFrame(
    pd.concat(
        [
            rice_training,
            nonrice_training,
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=REFERENCE_CRS,
)

validation_layer = gpd.GeoDataFrame(
    pd.concat(
        [
            rice_validation,
            nonrice_validation,
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=REFERENCE_CRS,
)

training_layer[
    "_feature_number"
] = np.arange(
    len(training_layer),
    dtype=int,
)

validation_layer[
    "_feature_number"
] = np.arange(
    len(validation_layer),
    dtype=int,
)

combined_layer = None

validation_design = (
    "Independent validation from four "
    "Tanore class-specific vector folders"
)

print(
    "Rice training:",
    rice_training_folder,
)

print(
    "Non-Rice training:",
    nonrice_training_folder,
)

print(
    "Rice validation:",
    rice_validation_folder,
)

print(
    "Non-Rice validation:",
    nonrice_validation_folder,
)

print()

print(
    "Training features:",
    len(training_layer),
    "| Rice:",
    int(
        (
            training_layer[
                "_class"
            ]
            == 1
        ).sum()
    ),
    "| Non-Rice:",
    int(
        (
            training_layer[
                "_class"
            ]
            == 0
        ).sum()
    ),
)

print(
    "Validation features:",
    len(validation_layer),
    "| Rice:",
    int(
        (
            validation_layer[
                "_class"
            ]
            == 1
        ).sum()
    ),
    "| Non-Rice:",
    int(
        (
            validation_layer[
                "_class"
            ]
            == 0
        ).sum()
    ),
)

print(
    "Validation design:",
    validation_design,
)


In [ ]:
# CELL 7 — Extract paired samples from both streams

rng = np.random.default_rng(RANDOM_SEED)


def spatial_block_id(x_value, y_value):
    return (
        f"{int(math.floor(x_value / SPATIAL_BLOCK_SIZE_METERS))}_"
        f"{int(math.floor(y_value / SPATIAL_BLOCK_SIZE_METERS))}"
    )


def extraction_geometry(geometry):
    if "point" in geometry.geom_type.lower():
        return geometry.buffer(POINT_BUFFER_METERS)

    return geometry


def extract_paired_samples(gdf, role):
    rows = {
        "FusedHybrid": [],
        "PlanetOnly": [],
    }

    skipped = []

    with ExitStack() as stack:
        handles = {
            stream_name: open_stream_handles(
                stack,
                stream_name,
            )
            for stream_name in STREAMS
        }

        reference = handles["PlanetOnly"][
            "bands"
        ]["Apr2"]

        for _, row in gdf.iterrows():
            geometry = extraction_geometry(
                row.geometry
            )

            raw_window = from_bounds(
                *geometry.bounds,
                transform=reference.transform,
            )

            window = clamp_window(
                raw_window,
                reference.width,
                reference.height,
            )

            if window is None:
                skipped.append({
                    "role": role,
                    "feature": int(
                        row["_feature_number"]
                    ),
                    "reason": "outside raster",
                })
                continue

            shape = (
                int(window.height),
                int(window.width),
            )

            window_mask = geometry_mask(
                [geometry],
                out_shape=shape,
                transform=rasterio.windows.transform(
                    window,
                    reference.transform,
                ),
                invert=True,
                all_touched=False,
            )

            stream_cubes = {}
            common_valid = window_mask.copy()

            for stream_name in STREAMS:
                cube, valid = read_feature_window(
                    handles[stream_name],
                    window,
                )

                stream_cubes[stream_name] = cube
                common_valid &= valid

            valid_indices = np.flatnonzero(
                common_valid.ravel()
            )

            if valid_indices.size == 0:
                skipped.append({
                    "role": role,
                    "feature": int(
                        row["_feature_number"]
                    ),
                    "reason": "no common valid pixels",
                })
                continue

            sample_size = min(
                MAX_PIXELS_PER_FEATURE,
                valid_indices.size,
            )

            selected = rng.choice(
                valid_indices,
                size=sample_size,
                replace=False,
            )

            selected_rows, selected_cols = np.unravel_index(
                selected,
                shape,
            )

            xs, ys = xy(
                rasterio.windows.transform(
                    window,
                    reference.transform,
                ),
                selected_rows,
                selected_cols,
                offset="center",
            )

            polygon_group = (
                f"{role}_feature_"
                f"{int(row['_feature_number'])}"
            )

            original_is_point = (
                "point"
                in row.geometry.geom_type.lower()
            )

            for local_index, (
                flat_index,
                x_value,
                y_value,
            ) in enumerate(
                zip(
                    selected,
                    xs,
                    ys,
                )
            ):
                sample_id = (
                    f"{role}_"
                    f"{int(row['_feature_number'])}_"
                    f"{local_index}"
                )

                block_id = spatial_block_id(
                    float(x_value),
                    float(y_value),
                )

                group_id = (
                    block_id
                    if original_is_point
                    else polygon_group
                )

                for stream_name in STREAMS:
                    feature_values = (
                        stream_cubes[stream_name]
                        .reshape(
                            len(FEATURE_NAMES),
                            -1,
                        )[:, flat_index]
                    )

                    record = {
                        "sample_id": sample_id,
                        "role": role,
                        "class": int(
                            row["_class"]
                        ),
                        "group": group_id,
                        "spatial_block": block_id,
                        "x": float(x_value),
                        "y": float(y_value),
                        "source_feature": int(
                            row["_feature_number"]
                        ),
                    }

                    record.update(
                        dict(
                            zip(
                                FEATURE_NAMES,
                                feature_values.astype(
                                    float
                                ),
                            )
                        )
                    )

                    rows[stream_name].append(
                        record
                    )

    output = {
        stream_name: pd.DataFrame(
            stream_rows
        )
        for stream_name, stream_rows
        in rows.items()
    }

    return output, pd.DataFrame(skipped)


if combined_layer is None:
    training_samples, skipped_train = (
        extract_paired_samples(
            training_layer,
            "train",
        )
    )

    validation_samples, skipped_validation = (
        extract_paired_samples(
            validation_layer,
            "validation",
        )
    )

else:
    combined_samples, skipped_combined = (
        extract_paired_samples(
            combined_layer,
            "combined",
        )
    )

    metadata = combined_samples[
        "FusedHybrid"
    ][
        [
            "sample_id",
            "class",
            "group",
        ]
    ].copy()

    selected_split = None

    for attempt in range(200):
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=HOLDOUT_FRACTION,
            random_state=RANDOM_SEED + attempt,
        )

        train_index, validation_index = next(
            splitter.split(
                metadata,
                metadata["class"],
                groups=metadata["group"],
            )
        )

        train_classes = set(
            metadata.iloc[
                train_index
            ]["class"].unique()
        )

        validation_classes = set(
            metadata.iloc[
                validation_index
            ]["class"].unique()
        )

        if (
            train_classes == {0, 1}
            and validation_classes == {0, 1}
        ):
            selected_split = (
                train_index,
                validation_index,
            )
            break

    if selected_split is None:
        raise ValueError(
            "A valid spatial holdout split containing both "
            "classes could not be created."
        )

    train_ids = set(
        metadata.iloc[
            selected_split[0]
        ]["sample_id"]
    )

    validation_ids = set(
        metadata.iloc[
            selected_split[1]
        ]["sample_id"]
    )

    training_samples = {}
    validation_samples = {}

    for stream_name in STREAMS:
        stream_df = combined_samples[
            stream_name
        ]

        training_samples[stream_name] = (
            stream_df[
                stream_df["sample_id"].isin(
                    train_ids
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

        validation_samples[stream_name] = (
            stream_df[
                stream_df["sample_id"].isin(
                    validation_ids
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

    skipped_train = skipped_combined
    skipped_validation = pd.DataFrame()


for stream_name in STREAMS:
    train_df = training_samples[stream_name]
    validation_df = validation_samples[
        stream_name
    ]

    if train_df.empty or validation_df.empty:
        raise ValueError(
            f"{stream_name}: training or validation samples are empty."
        )

    if set(train_df["class"].unique()) != {0, 1}:
        raise ValueError(
            f"{stream_name}: training samples must contain both classes."
        )

    if set(validation_df["class"].unique()) != {0, 1}:
        raise ValueError(
            f"{stream_name}: validation samples must contain both classes."
        )

    training_samples[stream_name].to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_Training_Samples.csv",
        index=False,
    )

    validation_samples[stream_name].to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_Validation_Samples.csv",
        index=False,
    )

sample_summary_rows = []

for stream_name in STREAMS:
    for split_name, frame in [
        ("Training", training_samples[stream_name]),
        (
            "Validation",
            validation_samples[stream_name],
        ),
    ]:
        for class_value, class_name in [
            (0, "Non-Boro"),
            (1, "Boro"),
        ]:
            subset = frame[
                frame["class"] == class_value
            ]

            sample_summary_rows.append({
                "stream": stream_name,
                "split": split_name,
                "class": class_name,
                "pixels": len(subset),
                "groups": subset["group"].nunique(),
                "spatial_blocks": subset[
                    "spatial_block"
                ].nunique(),
            })

sample_summary = pd.DataFrame(
    sample_summary_rows
)

display(sample_summary)

sample_summary.to_csv(
    TABLE_DIR
    / "Q1_Reference_Sample_Summary.csv",
    index=False,
)

print("✅ Paired samples extracted for both streams.")


In [ ]:
# CELL 8 — Descriptive figures and sample diagnostics

source_frames = []

if combined_layer is None:
    source_frames.append(
        training_layer.assign(
            plot_role="Training"
        )
    )

    source_frames.append(
        validation_layer.assign(
            plot_role="Validation"
        )
    )
else:
    source_frames.append(
        combined_layer.assign(
            plot_role="Combined layer"
        )
    )

sample_geometry = pd.concat(
    source_frames,
    ignore_index=True,
)

sample_geometry = gpd.GeoDataFrame(
    sample_geometry,
    geometry="geometry",
    crs=REFERENCE_CRS,
)

centroids = sample_geometry.copy()
centroids["geometry"] = centroids.geometry.centroid

figure, axis = plt.subplots(
    figsize=(8, 8)
)

centroids.plot(
    ax=axis,
    column="_class",
    categorical=True,
    legend=True,
    markersize=16,
)

axis.set_title(
    "Spatial distribution of reference samples"
)

axis.set_xlabel("Easting")
axis.set_ylabel("Northing")
axis.set_aspect("equal")

figure.tight_layout()
figure.savefig(
    FIGURE_DIR
    / "Q1_Reference_Sample_Distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(figure)


phenology_rows = []

for stream_name in STREAMS:
    frame = validation_samples[
        stream_name
    ]

    for class_value, class_name in [
        (0, "Non-Boro"),
        (1, "Boro"),
    ]:
        subset = frame[
            frame["class"] == class_value
        ]

        for date_name in DATES:
            values = subset[
                f"{date_name}_NDVI"
            ].to_numpy(dtype=float)

            mean_value = float(
                np.mean(values)
            )

            standard_error = float(
                np.std(
                    values,
                    ddof=1,
                )
                / math.sqrt(len(values))
            )

            phenology_rows.append({
                "stream": stream_name,
                "class": class_name,
                "date": date_name,
                "mean_ndvi": mean_value,
                "ci95_low": (
                    mean_value
                    - 1.96 * standard_error
                ),
                "ci95_high": (
                    mean_value
                    + 1.96 * standard_error
                ),
                "n": len(values),
            })

phenology_table = pd.DataFrame(
    phenology_rows
)

phenology_table.to_csv(
    TABLE_DIR
    / "Q1_NDVI_Phenology_Signatures.csv",
    index=False,
)

for stream_name in STREAMS:
    figure, axis = plt.subplots(
        figsize=(8, 5)
    )

    subset = phenology_table[
        phenology_table["stream"]
        == stream_name
    ]

    for class_name in [
        "Non-Boro",
        "Boro",
    ]:
        class_subset = (
            subset[
                subset["class"]
                == class_name
            ]
            .set_index("date")
            .loc[DATES]
            .reset_index()
        )

        x_values = np.arange(
            len(DATES)
        )

        axis.plot(
            x_values,
            class_subset["mean_ndvi"],
            marker="o",
            label=class_name,
        )

        axis.fill_between(
            x_values,
            class_subset["ci95_low"],
            class_subset["ci95_high"],
            alpha=0.2,
        )

    axis.set_xticks(
        np.arange(len(DATES)),
        DATES,
    )

    axis.set_ylabel("NDVI")
    axis.set_title(
        f"{stream_name}: validation NDVI phenology"
    )

    axis.legend()
    axis.grid(alpha=0.25)

    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR
        / f"Q1_{stream_name}_NDVI_Phenology.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)


for stream_name in STREAMS:
    frame = training_samples[
        stream_name
    ]

    correlation = frame[
        FEATURE_NAMES
    ].corr()

    correlation.to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_Feature_Correlation.csv"
    )

    figure, axis = plt.subplots(
        figsize=(12, 10)
    )

    image = axis.imshow(
        correlation,
        aspect="auto",
    )

    axis.set_xticks(
        np.arange(len(FEATURE_NAMES)),
        FEATURE_NAMES,
        rotation=90,
        fontsize=6,
    )

    axis.set_yticks(
        np.arange(len(FEATURE_NAMES)),
        FEATURE_NAMES,
        fontsize=6,
    )

    axis.set_title(
        f"{stream_name}: predictor correlation"
    )

    figure.colorbar(
        image,
        ax=axis,
        fraction=0.03,
    )

    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR
        / f"Q1_{stream_name}_Feature_Correlation.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

print("✅ Descriptive Q1 figures created.")


In [ ]:
# CELL 9 — Rule-based phenology classifier

def rule_binary_prediction(
    frame,
    thresholds,
):
    return (
        (
            frame["NDVI_max"].to_numpy()
            >= thresholds["peak_min"]
        )
        & (
            frame["NDVI_amplitude"].to_numpy()
            >= thresholds["amplitude_min"]
        )
        & (
            frame["dNDVI_Apr1_Jan"].to_numpy()
            >= thresholds["growth_min"]
        )
        & (
            frame["Mar_NDVI"].to_numpy()
            >= thresholds["march_min"]
        )
    ).astype(np.uint8)


def sigmoid(value):
    value = np.clip(
        value,
        -30,
        30,
    )

    return 1.0 / (
        1.0 + np.exp(-value)
    )


def rule_probability_score(
    frame,
    thresholds,
    scales,
):
    components = [
        sigmoid(
            (
                frame["NDVI_max"].to_numpy()
                - thresholds["peak_min"]
            )
            / scales["NDVI_max"]
        ),
        sigmoid(
            (
                frame[
                    "NDVI_amplitude"
                ].to_numpy()
                - thresholds[
                    "amplitude_min"
                ]
            )
            / scales[
                "NDVI_amplitude"
            ]
        ),
        sigmoid(
            (
                frame[
                    "dNDVI_Apr1_Jan"
                ].to_numpy()
                - thresholds[
                    "growth_min"
                ]
            )
            / scales[
                "dNDVI_Apr1_Jan"
            ]
        ),
        sigmoid(
            (
                frame[
                    "Mar_NDVI"
                ].to_numpy()
                - thresholds[
                    "march_min"
                ]
            )
            / scales["Mar_NDVI"]
        ),
    ]

    return np.mean(
        np.vstack(components),
        axis=0,
    )


def tune_rule_classifier(
    training_frame,
):
    y_true = training_frame[
        "class"
    ].to_numpy()

    quantiles = {
        "peak_min": np.unique(
            np.quantile(
                training_frame[
                    "NDVI_max"
                ],
                [0.20, 0.40, 0.60, 0.80],
            )
        ),
        "amplitude_min": np.unique(
            np.quantile(
                training_frame[
                    "NDVI_amplitude"
                ],
                [0.15, 0.35, 0.55, 0.75],
            )
        ),
        "growth_min": np.unique(
            np.quantile(
                training_frame[
                    "dNDVI_Apr1_Jan"
                ],
                [0.15, 0.35, 0.55, 0.75],
            )
        ),
        "march_min": np.unique(
            np.quantile(
                training_frame[
                    "Mar_NDVI"
                ],
                [0.20, 0.40, 0.60, 0.80],
            )
        ),
    }

    candidate_rows = []

    for peak_min in quantiles["peak_min"]:
        for amplitude_min in quantiles[
            "amplitude_min"
        ]:
            for growth_min in quantiles[
                "growth_min"
            ]:
                for march_min in quantiles[
                    "march_min"
                ]:
                    thresholds = {
                        "peak_min": float(
                            peak_min
                        ),
                        "amplitude_min": float(
                            amplitude_min
                        ),
                        "growth_min": float(
                            growth_min
                        ),
                        "march_min": float(
                            march_min
                        ),
                    }

                    prediction = (
                        rule_binary_prediction(
                            training_frame,
                            thresholds,
                        )
                    )

                    candidate_rows.append({
                        **thresholds,
                        "f1": f1_score(
                            y_true,
                            prediction,
                            zero_division=0,
                        ),
                        "balanced_accuracy": (
                            balanced_accuracy_score(
                                y_true,
                                prediction,
                            )
                        ),
                        "mcc": matthews_corrcoef(
                            y_true,
                            prediction,
                        ),
                    })

    candidates = pd.DataFrame(
        candidate_rows
    )

    candidates = candidates.sort_values(
        [
            "f1",
            "balanced_accuracy",
            "mcc",
        ],
        ascending=False,
    ).reset_index(drop=True)

    best_thresholds = {
        key: float(
            candidates.loc[0, key]
        )
        for key in [
            "peak_min",
            "amplitude_min",
            "growth_min",
            "march_min",
        ]
    }

    scales = {
        "NDVI_max": max(
            float(
                training_frame[
                    "NDVI_max"
                ].std()
            ),
            0.02,
        ),
        "NDVI_amplitude": max(
            float(
                training_frame[
                    "NDVI_amplitude"
                ].std()
            ),
            0.02,
        ),
        "dNDVI_Apr1_Jan": max(
            float(
                training_frame[
                    "dNDVI_Apr1_Jan"
                ].std()
            ),
            0.02,
        ),
        "Mar_NDVI": max(
            float(
                training_frame[
                    "Mar_NDVI"
                ].std()
            ),
            0.02,
        ),
    }

    return best_thresholds, scales, candidates


print("Rule-based functions are ready.")


In [ ]:
# CELL 10 — RF/XGBoost tuning helpers

def make_spatial_cv(y, groups):
    group_table = pd.DataFrame({
        "group": groups,
        "class": y,
    }).drop_duplicates()

    class_group_counts = (
        group_table.groupby(
            "class"
        )["group"].nunique()
    )

    if len(class_group_counts) < 2:
        raise ValueError(
            "Both classes need multiple independent groups."
        )

    n_splits = min(
        CV_SPLITS,
        int(class_group_counts.min()),
        int(
            group_table["group"].nunique()
        ),
    )

    if n_splits < 3:
        raise ValueError(
            "At least three spatial groups per class are "
            "required for spatial cross-validation."
        )

    return StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED,
    )


def optimal_f1_threshold(
    y_true,
    probability,
):
    precision, recall, thresholds = (
        precision_recall_curve(
            y_true,
            probability,
        )
    )

    if thresholds.size == 0:
        return 0.5

    f1_values = (
        2.0
        * precision[:-1]
        * recall[:-1]
        / np.maximum(
            precision[:-1]
            + recall[:-1],
            1e-12,
        )
    )

    best_index = int(
        np.nanargmax(f1_values)
    )

    return float(
        thresholds[best_index]
    )


def train_rf_model(
    X_train,
    y_train,
    groups,
):
    cv = make_spatial_cv(
        y_train,
        groups,
    )

    estimator = RandomForestClassifier(
        random_state=RANDOM_SEED,
        class_weight="balanced",
        n_jobs=-1,
    )

    parameter_distributions = {
        "n_estimators": [
            300,
            500,
            800,
            1000,
        ],
        "max_depth": [
            None,
            10,
            15,
            20,
            30,
        ],
        "min_samples_split": [
            2,
            5,
            10,
        ],
        "min_samples_leaf": [
            1,
            2,
            4,
            8,
        ],
        "max_features": [
            "sqrt",
            "log2",
            0.4,
            0.7,
        ],
        "bootstrap": [
            True,
            False,
        ],
    }

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=(
            parameter_distributions
        ),
        n_iter=RF_SEARCH_ITERATIONS,
        scoring="average_precision",
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_SEED,
        refit=True,
        return_train_score=True,
        verbose=1,
    )

    search.fit(
        X_train,
        y_train,
        groups=groups,
    )

    oof_probability = cross_val_predict(
        clone(search.best_estimator_),
        X_train,
        y_train,
        groups=groups,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    threshold = optimal_f1_threshold(
        y_train,
        oof_probability,
    )

    final_model = clone(
        search.best_estimator_
    )

    final_model.fit(
        X_train,
        y_train,
    )

    return (
        final_model,
        threshold,
        search,
        oof_probability,
    )


def train_xgb_model(
    X_train,
    y_train,
    groups,
):
    cv = make_spatial_cv(
        y_train,
        groups,
    )

    negative_count = max(
        int((y_train == 0).sum()),
        1,
    )

    positive_count = max(
        int((y_train == 1).sum()),
        1,
    )

    scale_pos_weight = (
        negative_count
        / positive_count
    )

    estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )

    parameter_distributions = {
        "n_estimators": [
            300,
            500,
            700,
            1000,
        ],
        "max_depth": [
            3,
            4,
            5,
            6,
            8,
        ],
        "learning_rate": [
            0.01,
            0.03,
            0.05,
            0.08,
            0.1,
        ],
        "subsample": [
            0.65,
            0.8,
            0.9,
            1.0,
        ],
        "colsample_bytree": [
            0.6,
            0.75,
            0.9,
            1.0,
        ],
        "min_child_weight": [
            1,
            3,
            5,
            8,
        ],
        "gamma": [
            0.0,
            0.1,
            0.3,
        ],
        "reg_alpha": [
            0.0,
            0.01,
            0.1,
            0.5,
        ],
        "reg_lambda": [
            0.5,
            1.0,
            2.0,
            5.0,
        ],
    }

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=(
            parameter_distributions
        ),
        n_iter=XGB_SEARCH_ITERATIONS,
        scoring="average_precision",
        n_jobs=-1,
        cv=cv,
        random_state=RANDOM_SEED,
        refit=True,
        return_train_score=True,
        verbose=1,
    )

    search.fit(
        X_train,
        y_train,
        groups=groups,
    )

    oof_probability = cross_val_predict(
        clone(search.best_estimator_),
        X_train,
        y_train,
        groups=groups,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    threshold = optimal_f1_threshold(
        y_train,
        oof_probability,
    )

    final_model = clone(
        search.best_estimator_
    )

    final_model.fit(
        X_train,
        y_train,
    )

    return (
        final_model,
        threshold,
        search,
        oof_probability,
    )


print("RF and XGBoost tuning functions are ready.")


In [ ]:
# CELL 11 — Train all three models for both streams

trained_models = {}
tuning_rows = []
rule_candidate_tables = {}

for stream_name in STREAMS:
    print()
    print("=" * 80)
    print("TRAINING STREAM:", stream_name)
    print("=" * 80)

    train_df = training_samples[
        stream_name
    ]

    X_train = train_df[
        FEATURE_NAMES
    ].to_numpy(
        dtype="float32"
    )

    y_train = train_df[
        "class"
    ].to_numpy(
        dtype=np.uint8
    )

    groups = train_df[
        "group"
    ].to_numpy()

    (
        rule_thresholds,
        rule_scales,
        rule_candidates,
    ) = tune_rule_classifier(
        train_df
    )

    rule_candidate_tables[
        stream_name
    ] = rule_candidates

    rule_candidates.to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_Rule_Threshold_Search.csv",
        index=False,
    )

    (
        rf_model,
        rf_threshold,
        rf_search,
        rf_oof_probability,
    ) = train_rf_model(
        X_train,
        y_train,
        groups,
    )

    (
        xgb_model,
        xgb_threshold,
        xgb_search,
        xgb_oof_probability,
    ) = train_xgb_model(
        X_train,
        y_train,
        groups,
    )

    trained_models[stream_name] = {
        "RuleBased": {
            "thresholds": rule_thresholds,
            "scales": rule_scales,
            "decision_threshold": 0.5,
        },
        "RandomForest": {
            "estimator": rf_model,
            "decision_threshold": rf_threshold,
        },
        "XGBoost": {
            "estimator": xgb_model,
            "decision_threshold": xgb_threshold,
        },
    }

    joblib.dump(
        trained_models[stream_name],
        MODEL_DIR
        / f"Q1_{stream_name}_Models.joblib",
    )

    rf_results = pd.DataFrame(
        rf_search.cv_results_
    )

    xgb_results = pd.DataFrame(
        xgb_search.cv_results_
    )

    rf_results.to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_RF_Tuning.csv",
        index=False,
    )

    xgb_results.to_csv(
        TABLE_DIR
        / f"Q1_{stream_name}_XGB_Tuning.csv",
        index=False,
    )

    tuning_rows.extend([
        {
            "stream": stream_name,
            "model": "RuleBased",
            "best_cv_average_precision": np.nan,
            "decision_threshold": 0.5,
            "best_parameters": json.dumps(
                rule_thresholds
            ),
        },
        {
            "stream": stream_name,
            "model": "RandomForest",
            "best_cv_average_precision": float(
                rf_search.best_score_
            ),
            "decision_threshold": float(
                rf_threshold
            ),
            "best_parameters": json.dumps(
                rf_search.best_params_
            ),
        },
        {
            "stream": stream_name,
            "model": "XGBoost",
            "best_cv_average_precision": float(
                xgb_search.best_score_
            ),
            "decision_threshold": float(
                xgb_threshold
            ),
            "best_parameters": json.dumps(
                xgb_search.best_params_
            ),
        },
    ])

    print("Rule thresholds:", rule_thresholds)
    print("RF threshold:", rf_threshold)
    print("XGB threshold:", xgb_threshold)

tuning_summary = pd.DataFrame(
    tuning_rows
)

display(tuning_summary)

tuning_summary.to_csv(
    TABLE_DIR
    / "Q1_Model_Tuning_and_Thresholds.csv",
    index=False,
)

print("✅ Six classifiers trained and saved.")


In [ ]:
# CELL 12 — Independent validation and 95% confidence intervals

def metric_dictionary(
    y_true,
    prediction,
    probability,
):
    matrix = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    )

    tn, fp, fn, tp = matrix.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "OA": accuracy_score(
            y_true,
            prediction,
        ),
        "Balanced_Accuracy": (
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "Precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Kappa": cohen_kappa_score(
            y_true,
            prediction,
        ),
        "MCC": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probability,
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probability,
        ),
        "Brier": brier_score_loss(
            y_true,
            probability,
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def cluster_bootstrap_ci(
    y_true,
    prediction,
    probability,
    groups,
    replicates=1000,
):
    frame = pd.DataFrame({
        "y": y_true,
        "pred": prediction,
        "prob": probability,
        "group": groups,
    })

    group_values = frame[
        "group"
    ].unique()

    group_indices = {
        group: frame.index[
            frame["group"] == group
        ].to_numpy()
        for group in group_values
    }

    generator = np.random.default_rng(
        RANDOM_SEED
    )

    metric_samples = {
        key: []
        for key in [
            "OA",
            "Balanced_Accuracy",
            "Precision",
            "Recall",
            "Specificity",
            "F1",
            "Kappa",
            "MCC",
            "ROC_AUC",
            "PR_AUC",
            "Brier",
        ]
    }

    successful = 0
    attempts = 0

    while (
        successful < replicates
        and attempts < replicates * 5
    ):
        attempts += 1

        sampled_groups = generator.choice(
            group_values,
            size=len(group_values),
            replace=True,
        )

        sampled_indices = np.concatenate([
            group_indices[group]
            for group in sampled_groups
        ])

        sample = frame.loc[
            sampled_indices
        ]

        if sample["y"].nunique() < 2:
            continue

        metrics = metric_dictionary(
            sample["y"].to_numpy(),
            sample["pred"].to_numpy(),
            sample["prob"].to_numpy(),
        )

        for key in metric_samples:
            metric_samples[key].append(
                metrics[key]
            )

        successful += 1

    confidence_intervals = {}

    for key, values in metric_samples.items():
        values = np.asarray(
            values,
            dtype=float,
        )

        confidence_intervals[key] = (
            float(
                np.nanpercentile(
                    values,
                    2.5,
                )
            ),
            float(
                np.nanpercentile(
                    values,
                    97.5,
                )
            ),
        )

    return confidence_intervals


performance_rows = []
prediction_rows = []

for stream_name in STREAMS:
    validation_df = validation_samples[
        stream_name
    ]

    X_validation = validation_df[
        FEATURE_NAMES
    ].to_numpy(
        dtype="float32"
    )

    y_validation = validation_df[
        "class"
    ].to_numpy(
        dtype=np.uint8
    )

    validation_groups = (
        validation_df["group"]
        .to_numpy()
    )

    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        model_info = trained_models[
            stream_name
        ][model_name]

        if model_name == "RuleBased":
            probability = (
                rule_probability_score(
                    validation_df,
                    model_info["thresholds"],
                    model_info["scales"],
                )
            )

            prediction = (
                rule_binary_prediction(
                    validation_df,
                    model_info["thresholds"],
                )
            )

        else:
            estimator = model_info[
                "estimator"
            ]

            probability = estimator.predict_proba(
                X_validation
            )[:, 1]

            prediction = (
                probability
                >= model_info[
                    "decision_threshold"
                ]
            ).astype(np.uint8)

        point_metrics = metric_dictionary(
            y_validation,
            prediction,
            probability,
        )

        confidence_intervals = (
            cluster_bootstrap_ci(
                y_validation,
                prediction,
                probability,
                validation_groups,
                replicates=(
                    BOOTSTRAP_REPLICATES
                ),
            )
        )

        row = {
            "stream": stream_name,
            "model": model_name,
            "validation_pixels": len(
                y_validation
            ),
            "validation_groups": (
                pd.Series(
                    validation_groups
                ).nunique()
            ),
            "decision_threshold": (
                model_info[
                    "decision_threshold"
                ]
            ),
            **point_metrics,
        }

        for metric_name, (
            lower,
            upper,
        ) in confidence_intervals.items():
            row[
                f"{metric_name}_CI95_Low"
            ] = lower

            row[
                f"{metric_name}_CI95_High"
            ] = upper

        performance_rows.append(row)

        for index in range(
            len(validation_df)
        ):
            prediction_rows.append({
                "sample_id": (
                    validation_df.iloc[
                        index
                    ]["sample_id"]
                ),
                "group": (
                    validation_df.iloc[
                        index
                    ]["group"]
                ),
                "x": float(
                    validation_df.iloc[
                        index
                    ]["x"]
                ),
                "y": float(
                    validation_df.iloc[
                        index
                    ]["y"]
                ),
                "true_class": int(
                    y_validation[index]
                ),
                "stream": stream_name,
                "model": model_name,
                "probability": float(
                    probability[index]
                ),
                "prediction": int(
                    prediction[index]
                ),
            })

performance_table = pd.DataFrame(
    performance_rows
)

validation_predictions = pd.DataFrame(
    prediction_rows
)

display(
    performance_table[
        [
            "stream",
            "model",
            "OA",
            "Balanced_Accuracy",
            "Precision",
            "Recall",
            "Specificity",
            "F1",
            "Kappa",
            "MCC",
            "ROC_AUC",
            "PR_AUC",
            "Brier",
        ]
    ].round(4)
)

performance_table.to_csv(
    TABLE_DIR
    / "Q1_Independent_Validation_Performance.csv",
    index=False,
)

validation_predictions.to_csv(
    TABLE_DIR
    / "Q1_Validation_Predictions.csv",
    index=False,
)

print("✅ Independent validation and confidence intervals complete.")


In [ ]:
# CELL 13 — Confusion matrices, ROC and PR figures

for stream_name in STREAMS:
    stream_predictions = (
        validation_predictions[
            validation_predictions[
                "stream"
            ] == stream_name
        ]
    )

    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        subset = stream_predictions[
            stream_predictions[
                "model"
            ] == model_name
        ]

        y_true = subset[
            "true_class"
        ].to_numpy()

        prediction = subset[
            "prediction"
        ].to_numpy()

        matrix = confusion_matrix(
            y_true,
            prediction,
            labels=[0, 1],
        )

        pd.DataFrame(
            matrix,
            index=[
                "True_NonBoro",
                "True_Boro",
            ],
            columns=[
                "Pred_NonBoro",
                "Pred_Boro",
            ],
        ).to_csv(
            TABLE_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Confusion_Matrix.csv"
            )
        )

        figure, axis = plt.subplots(
            figsize=(5, 5)
        )

        display_object = (
            ConfusionMatrixDisplay(
                confusion_matrix=matrix,
                display_labels=[
                    "Non-Boro",
                    "Boro",
                ],
            )
        )

        display_object.plot(
            ax=axis,
            values_format="d",
            colorbar=False,
        )

        axis.set_title(
            f"{stream_name} — {model_name}"
        )

        figure.tight_layout()
        figure.savefig(
            FIGURE_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Confusion_Matrix.png"
            ),
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(figure)

        normalized_matrix = (
            matrix
            / np.maximum(
                matrix.sum(
                    axis=1,
                    keepdims=True,
                ),
                1,
            )
        )

        pd.DataFrame(
            normalized_matrix,
            index=[
                "True_NonBoro",
                "True_Boro",
            ],
            columns=[
                "Pred_NonBoro",
                "Pred_Boro",
            ],
        ).to_csv(
            TABLE_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Normalized_Confusion_Matrix.csv"
            )
        )

    figure, axis = plt.subplots(
        figsize=(7, 6)
    )

    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        subset = stream_predictions[
            stream_predictions[
                "model"
            ] == model_name
        ]

        false_positive_rate, true_positive_rate, _ = (
            roc_curve(
                subset["true_class"],
                subset["probability"],
            )
        )

        auc_value = roc_auc_score(
            subset["true_class"],
            subset["probability"],
        )

        axis.plot(
            false_positive_rate,
            true_positive_rate,
            label=(
                f"{model_name} "
                f"(AUC={auc_value:.3f})"
            ),
        )

    axis.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="No-skill",
    )

    axis.set_xlabel("False-positive rate")
    axis.set_ylabel("True-positive rate")
    axis.set_title(
        f"{stream_name}: ROC curves"
    )
    axis.legend()
    axis.grid(alpha=0.25)

    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR
        / f"Q1_{stream_name}_ROC_Curves.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

    figure, axis = plt.subplots(
        figsize=(7, 6)
    )

    prevalence = float(
        stream_predictions[
            stream_predictions[
                "model"
            ] == "RandomForest"
        ]["true_class"].mean()
    )

    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        subset = stream_predictions[
            stream_predictions[
                "model"
            ] == model_name
        ]

        precision, recall, _ = (
            precision_recall_curve(
                subset["true_class"],
                subset["probability"],
            )
        )

        ap_value = average_precision_score(
            subset["true_class"],
            subset["probability"],
        )

        axis.plot(
            recall,
            precision,
            label=(
                f"{model_name} "
                f"(AP={ap_value:.3f})"
            ),
        )

    axis.axhline(
        prevalence,
        linestyle="--",
        label="Prevalence baseline",
    )

    axis.set_xlabel("Recall")
    axis.set_ylabel("Precision")
    axis.set_title(
        f"{stream_name}: precision-recall curves"
    )
    axis.legend()
    axis.grid(alpha=0.25)

    figure.tight_layout()
    figure.savefig(
        FIGURE_DIR
        / f"Q1_{stream_name}_PR_Curves.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)

print("✅ Confusion matrices, ROC and PR figures created.")


In [ ]:
# CELL 14 — Paired statistical model and stream comparisons

def exact_mcnemar(
    truth,
    prediction_a,
    prediction_b,
):
    correct_a = prediction_a == truth
    correct_b = prediction_b == truth

    b_value = int(
        np.sum(
            correct_a
            & ~correct_b
        )
    )

    c_value = int(
        np.sum(
            ~correct_a
            & correct_b
        )
    )

    discordant = b_value + c_value

    p_value = (
        binomtest(
            min(b_value, c_value),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        ).pvalue
        if discordant > 0
        else 1.0
    )

    return {
        "A_correct_B_wrong": b_value,
        "A_wrong_B_correct": c_value,
        "discordant_pairs": discordant,
        "exact_p_value": float(
            p_value
        ),
    }


def paired_group_bootstrap_f1_difference(
    merged,
    replicates=1000,
):
    groups = merged[
        "group"
    ].unique()

    group_indices = {
        group: merged.index[
            merged["group"] == group
        ].to_numpy()
        for group in groups
    }

    generator = np.random.default_rng(
        RANDOM_SEED
    )

    differences = []

    for _ in range(replicates):
        selected_groups = generator.choice(
            groups,
            size=len(groups),
            replace=True,
        )

        indices = np.concatenate([
            group_indices[group]
            for group in selected_groups
        ])

        sample = merged.loc[
            indices
        ]

        f1_a = f1_score(
            sample["true_class"],
            sample["prediction_a"],
            zero_division=0,
        )

        f1_b = f1_score(
            sample["true_class"],
            sample["prediction_b"],
            zero_division=0,
        )

        differences.append(
            f1_a - f1_b
        )

    return {
        "delta_f1_A_minus_B": float(
            np.mean(differences)
        ),
        "delta_f1_CI95_Low": float(
            np.percentile(
                differences,
                2.5,
            )
        ),
        "delta_f1_CI95_High": float(
            np.percentile(
                differences,
                97.5,
            )
        ),
    }


comparison_rows = []

model_names = [
    "RuleBased",
    "RandomForest",
    "XGBoost",
]

for stream_name in STREAMS:
    for first_index in range(
        len(model_names)
    ):
        for second_index in range(
            first_index + 1,
            len(model_names),
        ):
            model_a = model_names[
                first_index
            ]

            model_b = model_names[
                second_index
            ]

            first = validation_predictions[
                (
                    validation_predictions[
                        "stream"
                    ] == stream_name
                )
                & (
                    validation_predictions[
                        "model"
                    ] == model_a
                )
            ][
                [
                    "sample_id",
                    "group",
                    "true_class",
                    "prediction",
                ]
            ].rename(
                columns={
                    "prediction": (
                        "prediction_a"
                    )
                }
            )

            second = validation_predictions[
                (
                    validation_predictions[
                        "stream"
                    ] == stream_name
                )
                & (
                    validation_predictions[
                        "model"
                    ] == model_b
                )
            ][
                [
                    "sample_id",
                    "prediction",
                ]
            ].rename(
                columns={
                    "prediction": (
                        "prediction_b"
                    )
                }
            )

            merged = first.merge(
                second,
                on="sample_id",
                how="inner",
            )

            comparison_rows.append({
                "comparison_type": (
                    "Within-stream model comparison"
                ),
                "stream_or_model": (
                    stream_name
                ),
                "A": model_a,
                "B": model_b,
                **exact_mcnemar(
                    merged["true_class"].to_numpy(),
                    merged[
                        "prediction_a"
                    ].to_numpy(),
                    merged[
                        "prediction_b"
                    ].to_numpy(),
                ),
                **paired_group_bootstrap_f1_difference(
                    merged,
                    replicates=(
                        BOOTSTRAP_REPLICATES
                    ),
                ),
            })


for model_name in model_names:
    first = validation_predictions[
        (
            validation_predictions[
                "stream"
            ] == "FusedHybrid"
        )
        & (
            validation_predictions[
                "model"
            ] == model_name
        )
    ][
        [
            "sample_id",
            "group",
            "true_class",
            "prediction",
        ]
    ].rename(
        columns={
            "prediction": "prediction_a"
        }
    )

    second = validation_predictions[
        (
            validation_predictions[
                "stream"
            ] == "PlanetOnly"
        )
        & (
            validation_predictions[
                "model"
            ] == model_name
        )
    ][
        [
            "sample_id",
            "prediction",
        ]
    ].rename(
        columns={
            "prediction": "prediction_b"
        }
    )

    merged = first.merge(
        second,
        on="sample_id",
        how="inner",
    )

    comparison_rows.append({
        "comparison_type": (
            "Between-stream comparison"
        ),
        "stream_or_model": model_name,
        "A": "FusedHybrid",
        "B": "PlanetOnly",
        **exact_mcnemar(
            merged["true_class"].to_numpy(),
            merged[
                "prediction_a"
            ].to_numpy(),
            merged[
                "prediction_b"
            ].to_numpy(),
        ),
        **paired_group_bootstrap_f1_difference(
            merged,
            replicates=(
                BOOTSTRAP_REPLICATES
            ),
        ),
    })

pairwise_comparisons = pd.DataFrame(
    comparison_rows
)

display(pairwise_comparisons.round(5))

pairwise_comparisons.to_csv(
    TABLE_DIR
    / "Q1_Paired_Model_and_Stream_Comparisons.csv",
    index=False,
)

print("✅ McNemar and paired bootstrap comparisons complete.")


In [ ]:
# CELL 15 — RF importance, XGBoost importance and SHAP contributions

rf_importance_rows = []
xgb_importance_rows = []
shap_rows = []

for stream_name in STREAMS:
    validation_df = validation_samples[
        stream_name
    ]

    X_validation = validation_df[
        FEATURE_NAMES
    ].to_numpy(
        dtype="float32"
    )

    y_validation = validation_df[
        "class"
    ].to_numpy(
        dtype=np.uint8
    )

    generator = np.random.default_rng(
        RANDOM_SEED
    )

    maximum_explain_samples = min(
        5000,
        len(X_validation),
    )

    selected = generator.choice(
        len(X_validation),
        size=maximum_explain_samples,
        replace=False,
    )

    X_explain = X_validation[
        selected
    ]

    y_explain = y_validation[
        selected
    ]

    rf_model = trained_models[
        stream_name
    ]["RandomForest"]["estimator"]

    rf_permutation = permutation_importance(
        rf_model,
        X_explain,
        y_explain,
        scoring="average_precision",
        n_repeats=15,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    rf_table = pd.DataFrame({
        "stream": stream_name,
        "feature": FEATURE_NAMES,
        "importance_mean": (
            rf_permutation.importances_mean
        ),
        "importance_std": (
            rf_permutation.importances_std
        ),
    }).sort_values(
        "importance_mean",
        ascending=False,
    )

    rf_importance_rows.extend(
        rf_table.to_dict(
            orient="records"
        )
    )

    xgb_model = trained_models[
        stream_name
    ]["XGBoost"]["estimator"]

    gain_table = pd.DataFrame({
        "stream": stream_name,
        "feature": FEATURE_NAMES,
        "gain_importance": (
            xgb_model.feature_importances_
        ),
    }).sort_values(
        "gain_importance",
        ascending=False,
    )

    xgb_importance_rows.extend(
        gain_table.to_dict(
            orient="records"
        )
    )

    contributions = (
        xgb_model.get_booster().predict(
            DMatrix(X_explain),
            pred_contribs=True,
        )
    )

    mean_absolute_shap = np.mean(
        np.abs(
            contributions[:, :-1]
        ),
        axis=0,
    )

    shap_table = pd.DataFrame({
        "stream": stream_name,
        "feature": FEATURE_NAMES,
        "mean_absolute_shap": (
            mean_absolute_shap
        ),
    }).sort_values(
        "mean_absolute_shap",
        ascending=False,
    )

    shap_rows.extend(
        shap_table.to_dict(
            orient="records"
        )
    )

    for table, value_column, title, filename in [
        (
            rf_table.head(20),
            "importance_mean",
            (
                f"{stream_name}: RF permutation "
                "importance"
            ),
            (
                f"Q1_{stream_name}_RF_"
                "Permutation_Importance.png"
            ),
        ),
        (
            gain_table.head(20),
            "gain_importance",
            (
                f"{stream_name}: XGBoost gain "
                "importance"
            ),
            (
                f"Q1_{stream_name}_XGB_"
                "Gain_Importance.png"
            ),
        ),
        (
            shap_table.head(20),
            "mean_absolute_shap",
            (
                f"{stream_name}: XGBoost mean "
                "absolute SHAP"
            ),
            (
                f"Q1_{stream_name}_XGB_SHAP.png"
            ),
        ),
    ]:
        plotting_table = table.sort_values(
            value_column,
            ascending=True,
        )

        figure, axis = plt.subplots(
            figsize=(8, 7)
        )

        axis.barh(
            plotting_table["feature"],
            plotting_table[
                value_column
            ],
        )

        axis.set_title(title)
        axis.set_xlabel(value_column)

        figure.tight_layout()
        figure.savefig(
            FIGURE_DIR / filename,
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(figure)

rf_importance_table = pd.DataFrame(
    rf_importance_rows
)

xgb_importance_table = pd.DataFrame(
    xgb_importance_rows
)

shap_importance_table = pd.DataFrame(
    shap_rows
)

rf_importance_table.to_csv(
    TABLE_DIR
    / "Q1_RF_Permutation_Importance.csv",
    index=False,
)

xgb_importance_table.to_csv(
    TABLE_DIR
    / "Q1_XGB_Gain_Importance.csv",
    index=False,
)

shap_importance_table.to_csv(
    TABLE_DIR
    / "Q1_XGB_SHAP_Importance.csv",
    index=False,
)

print("✅ Explainability tables and figures created.")


In [ ]:
# CELL 16 — Blockwise classification maps

def feature_matrix_to_frame(
    feature_matrix,
):
    return pd.DataFrame(
        feature_matrix,
        columns=FEATURE_NAMES,
    )


def predict_three_models(
    stream_name,
    feature_matrix,
):
    output = {}

    feature_frame = (
        feature_matrix_to_frame(
            feature_matrix
        )
    )

    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        model_info = trained_models[
            stream_name
        ][model_name]

        if model_name == "RuleBased":
            probability = (
                rule_probability_score(
                    feature_frame,
                    model_info["thresholds"],
                    model_info["scales"],
                )
            )

            prediction = (
                rule_binary_prediction(
                    feature_frame,
                    model_info["thresholds"],
                )
            )

        else:
            estimator = model_info[
                "estimator"
            ]

            probability = estimator.predict_proba(
                feature_matrix
            )[:, 1]

            prediction = (
                probability
                >= model_info[
                    "decision_threshold"
                ]
            ).astype(np.uint8)

        uncertainty = (
            1.0
            - 2.0
            * np.abs(
                probability - 0.5
            )
        )

        output[model_name] = {
            "prediction": prediction,
            "probability": probability.astype(
                "float32"
            ),
            "uncertainty": uncertainty.astype(
                "float32"
            ),
        }

    return output


map_inventory_rows = []
area_rows = []

for stream_name in STREAMS:
    print()
    print("=" * 80)
    print("MAPPING STREAM:", stream_name)
    print("=" * 80)

    with ExitStack() as stack:
        handles = open_stream_handles(
            stack,
            stream_name,
        )

        reference = handles[
            "bands"
        ]["Apr2"]

        class_profile = (
            reference.profile.copy()
        )

        class_profile.update(
            driver="GTiff",
            count=1,
            dtype="uint8",
            nodata=CLASS_NODATA,
            compress="DEFLATE",
            predictor=2,
            tiled=True,
            blockxsize=MAP_BLOCK_SIZE,
            blockysize=MAP_BLOCK_SIZE,
            BIGTIFF="IF_SAFER",
        )

        float_profile = (
            reference.profile.copy()
        )

        float_profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=FLOAT_NODATA,
            compress="DEFLATE",
            predictor=3,
            tiled=True,
            blockxsize=MAP_BLOCK_SIZE,
            blockysize=MAP_BLOCK_SIZE,
            BIGTIFF="IF_SAFER",
        )

        destinations = {}

        for model_name in [
            "RuleBased",
            "RandomForest",
            "XGBoost",
        ]:
            class_path = (
                MAP_DIR
                / (
                    f"Q1_{stream_name}_{model_name}"
                    "_Classification.tif"
                )
            )

            probability_path = (
                MAP_DIR
                / (
                    f"Q1_{stream_name}_{model_name}"
                    "_Probability.tif"
                )
            )

            uncertainty_path = (
                MAP_DIR
                / (
                    f"Q1_{stream_name}_{model_name}"
                    "_Uncertainty.tif"
                )
            )

            destinations[model_name] = {
                "class_path": class_path,
                "probability_path": (
                    probability_path
                ),
                "uncertainty_path": (
                    uncertainty_path
                ),
                "class_dst": stack.enter_context(
                    rasterio.open(
                        class_path,
                        "w",
                        **class_profile,
                    )
                ),
                "probability_dst": (
                    stack.enter_context(
                        rasterio.open(
                            probability_path,
                            "w",
                            **float_profile,
                        )
                    )
                ),
                "uncertainty_dst": (
                    stack.enter_context(
                        rasterio.open(
                            uncertainty_path,
                            "w",
                            **float_profile,
                        )
                    )
                ),
                "rice_pixels": 0,
                "nonrice_pixels": 0,
                "valid_pixels": 0,
            }

        for window in iter_windows(
            reference.width,
            reference.height,
            MAP_BLOCK_SIZE,
        ):
            feature_cube, valid = (
                read_feature_window(
                    handles,
                    window,
                )
            )

            valid_indices = np.flatnonzero(
                valid.ravel()
            )

            if valid_indices.size > 0:
                feature_matrix = (
                    feature_cube
                    .reshape(
                        len(FEATURE_NAMES),
                        -1,
                    )[:, valid_indices]
                    .T
                    .astype("float32")
                )

                predictions = (
                    predict_three_models(
                        stream_name,
                        feature_matrix,
                    )
                )
            else:
                predictions = {}

            output_shape = (
                int(window.height),
                int(window.width),
            )

            for model_name in [
                "RuleBased",
                "RandomForest",
                "XGBoost",
            ]:
                class_array = np.full(
                    output_shape,
                    CLASS_NODATA,
                    dtype=np.uint8,
                )

                probability_array = np.full(
                    output_shape,
                    FLOAT_NODATA,
                    dtype="float32",
                )

                uncertainty_array = np.full(
                    output_shape,
                    FLOAT_NODATA,
                    dtype="float32",
                )

                if valid_indices.size > 0:
                    class_array.ravel()[
                        valid_indices
                    ] = predictions[
                        model_name
                    ]["prediction"]

                    probability_array.ravel()[
                        valid_indices
                    ] = predictions[
                        model_name
                    ]["probability"]

                    uncertainty_array.ravel()[
                        valid_indices
                    ] = predictions[
                        model_name
                    ]["uncertainty"]

                destination = destinations[
                    model_name
                ]

                destination[
                    "class_dst"
                ].write(
                    class_array,
                    1,
                    window=window,
                )

                destination[
                    "probability_dst"
                ].write(
                    probability_array,
                    1,
                    window=window,
                )

                destination[
                    "uncertainty_dst"
                ].write(
                    uncertainty_array,
                    1,
                    window=window,
                )

                destination[
                    "rice_pixels"
                ] += int(
                    np.sum(
                        class_array == 1
                    )
                )

                destination[
                    "nonrice_pixels"
                ] += int(
                    np.sum(
                        class_array == 0
                    )
                )

                destination[
                    "valid_pixels"
                ] += int(
                    np.sum(
                        class_array
                        != CLASS_NODATA
                    )
                )

        pixel_area_square_meters = abs(
            reference.transform.a
            * reference.transform.e
        )

        for model_name, destination in (
            destinations.items()
        ):
            map_inventory_rows.extend([
                {
                    "stream": stream_name,
                    "model": model_name,
                    "product": "Classification",
                    "path": str(
                        destination[
                            "class_path"
                        ]
                    ),
                },
                {
                    "stream": stream_name,
                    "model": model_name,
                    "product": "Probability",
                    "path": str(
                        destination[
                            "probability_path"
                        ]
                    ),
                },
                {
                    "stream": stream_name,
                    "model": model_name,
                    "product": "Uncertainty",
                    "path": str(
                        destination[
                            "uncertainty_path"
                        ]
                    ),
                },
            ])

            rice_area = (
                destination[
                    "rice_pixels"
                ]
                * pixel_area_square_meters
                / 1_000_000.0
            )

            nonrice_area = (
                destination[
                    "nonrice_pixels"
                ]
                * pixel_area_square_meters
                / 1_000_000.0
            )

            valid_area = (
                destination[
                    "valid_pixels"
                ]
                * pixel_area_square_meters
                / 1_000_000.0
            )

            area_rows.append({
                "stream": stream_name,
                "model": model_name,
                "boro_area_km2": rice_area,
                "nonboro_area_km2": nonrice_area,
                "valid_mapped_area_km2": valid_area,
                "boro_percent_of_valid": (
                    100.0
                    * rice_area
                    / valid_area
                    if valid_area > 0
                    else np.nan
                ),
                "note": (
                    "Mapped area only; not an "
                    "accuracy-adjusted area estimate."
                ),
            })

map_inventory = pd.DataFrame(
    map_inventory_rows
)

mapped_area_table = pd.DataFrame(
    area_rows
)

map_inventory.to_csv(
    TABLE_DIR
    / "Q1_Map_Product_Inventory.csv",
    index=False,
)

mapped_area_table.to_csv(
    TABLE_DIR
    / "Q1_Mapped_Area_Summary.csv",
    index=False,
)

display(mapped_area_table.round(4))

print("✅ Classification, probability and uncertainty GeoTIFFs created.")


In [ ]:
# CELL 17 — Publication map figures

def add_north_arrow(axis):
    axis.annotate(
        "N",
        xy=(0.94, 0.93),
        xytext=(0.94, 0.78),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=12,
        arrowprops={
            "arrowstyle": "-|>",
            "linewidth": 1.5,
        },
    )


def add_scale_bar(
    axis,
    bounds,
):
    width = bounds.right - bounds.left
    scale_length = (
        round(width / 5 / 1000)
        * 1000
    )

    if scale_length <= 0:
        return

    x_start = (
        bounds.left
        + width * 0.08
    )

    y_start = (
        bounds.bottom
        + (
            bounds.top
            - bounds.bottom
        )
        * 0.06
    )

    axis.plot(
        [
            x_start,
            x_start + scale_length,
        ],
        [y_start, y_start],
        linewidth=3,
    )

    axis.text(
        x_start + scale_length / 2,
        y_start,
        f"{scale_length / 1000:.0f} km",
        ha="center",
        va="bottom",
    )


def read_quicklook(
    path,
    max_dimension=1400,
):
    with rasterio.open(path) as src:
        scale = max(
            src.width / max_dimension,
            src.height / max_dimension,
            1,
        )

        output_width = max(
            1,
            int(src.width / scale),
        )

        output_height = max(
            1,
            int(src.height / scale),
        )

        data = src.read(
            1,
            out_shape=(
                output_height,
                output_width,
            ),
            resampling=(
                rasterio.enums.Resampling.nearest
            ),
        )

        return data, src.bounds, src.nodata


for stream_name in STREAMS:
    for model_name in [
        "RuleBased",
        "RandomForest",
        "XGBoost",
    ]:
        class_path = (
            MAP_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Classification.tif"
            )
        )

        probability_path = (
            MAP_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Probability.tif"
            )
        )

        data, bounds, nodata = read_quicklook(
            class_path
        )

        masked = np.ma.masked_equal(
            data,
            CLASS_NODATA,
        )

        figure, axis = plt.subplots(
            figsize=(8, 9)
        )

        cmap = plt.get_cmap(
            "viridis",
            2,
        )

        axis.imshow(
            masked,
            extent=[
                bounds.left,
                bounds.right,
                bounds.bottom,
                bounds.top,
            ],
            origin="upper",
            cmap=cmap,
            vmin=0,
            vmax=1,
        )

        legend_handles = [
            Patch(
                facecolor=cmap(0),
                label="Non-Boro",
            ),
            Patch(
                facecolor=cmap(1),
                label="Boro rice",
            ),
        ]

        axis.legend(
            handles=legend_handles,
            loc="lower right",
        )

        axis.set_title(
            f"{stream_name} — {model_name}"
        )

        axis.set_xlabel("Easting")
        axis.set_ylabel("Northing")
        axis.set_aspect("equal")
        add_north_arrow(axis)
        add_scale_bar(axis, bounds)

        figure.tight_layout()
        figure.savefig(
            FIGURE_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Classification_Map.png"
            ),
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(figure)

        probability, bounds, nodata = (
            read_quicklook(
                probability_path
            )
        )

        probability = np.ma.masked_equal(
            probability,
            FLOAT_NODATA,
        )

        figure, axis = plt.subplots(
            figsize=(8, 9)
        )

        image = axis.imshow(
            probability,
            extent=[
                bounds.left,
                bounds.right,
                bounds.bottom,
                bounds.top,
            ],
            origin="upper",
            vmin=0,
            vmax=1,
        )

        figure.colorbar(
            image,
            ax=axis,
            fraction=0.035,
            label="Boro probability",
        )

        axis.set_title(
            f"{stream_name} — {model_name} probability"
        )

        axis.set_xlabel("Easting")
        axis.set_ylabel("Northing")
        axis.set_aspect("equal")
        add_north_arrow(axis)
        add_scale_bar(axis, bounds)

        figure.tight_layout()
        figure.savefig(
            FIGURE_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Probability_Map.png"
            ),
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(figure)

print("✅ 300-DPI classification and probability maps created.")


In [ ]:
# CELL 18 — Consensus and fused-versus-Planet disagreement maps

agreement_rows = []
consensus_rows = []

for stream_name in STREAMS:
    class_paths = {
        model_name: (
            MAP_DIR
            / (
                f"Q1_{stream_name}_{model_name}"
                "_Classification.tif"
            )
        )
        for model_name in [
            "RuleBased",
            "RandomForest",
            "XGBoost",
        ]
    }

    consensus_path = (
        MAP_DIR
        / f"Q1_{stream_name}_Consensus_Classification.tif"
    )

    disagreement_count_path = (
        MAP_DIR
        / f"Q1_{stream_name}_Model_Disagreement_Count.tif"
    )

    with ExitStack() as stack:
        sources = {
            model_name: stack.enter_context(
                rasterio.open(path)
            )
            for model_name, path
            in class_paths.items()
        }

        reference = sources[
            "RandomForest"
        ]

        profile = reference.profile.copy()
        profile.update(
            dtype="uint8",
            nodata=CLASS_NODATA,
            compress="DEFLATE",
            tiled=True,
            blockxsize=MAP_BLOCK_SIZE,
            blockysize=MAP_BLOCK_SIZE,
        )

        consensus_dst = stack.enter_context(
            rasterio.open(
                consensus_path,
                "w",
                **profile,
            )
        )

        disagreement_dst = stack.enter_context(
            rasterio.open(
                disagreement_count_path,
                "w",
                **profile,
            )
        )

        consensus_rice = 0
        consensus_valid = 0

        for window in iter_windows(
            reference.width,
            reference.height,
            MAP_BLOCK_SIZE,
        ):
            arrays = np.stack([
                sources[
                    model_name
                ].read(
                    1,
                    window=window,
                )
                for model_name in [
                    "RuleBased",
                    "RandomForest",
                    "XGBoost",
                ]
            ])

            valid = np.all(
                arrays != CLASS_NODATA,
                axis=0,
            )

            vote_sum = np.sum(
                arrays == 1,
                axis=0,
            )

            consensus = np.full(
                vote_sum.shape,
                CLASS_NODATA,
                dtype=np.uint8,
            )

            consensus[valid] = (
                vote_sum[valid] >= 2
            ).astype(np.uint8)

            disagreement = np.full(
                vote_sum.shape,
                CLASS_NODATA,
                dtype=np.uint8,
            )

            disagreement[valid] = np.minimum(
                vote_sum[valid],
                3 - vote_sum[valid],
            ).astype(np.uint8)

            consensus_dst.write(
                consensus,
                1,
                window=window,
            )

            disagreement_dst.write(
                disagreement,
                1,
                window=window,
            )

            consensus_rice += int(
                np.sum(
                    consensus == 1
                )
            )

            consensus_valid += int(
                np.sum(valid)
            )

        consensus_rows.append({
            "stream": stream_name,
            "consensus_path": str(
                consensus_path
            ),
            "model_disagreement_path": str(
                disagreement_count_path
            ),
            "consensus_boro_pixels": (
                consensus_rice
            ),
            "consensus_valid_pixels": (
                consensus_valid
            ),
        })


for model_name in [
    "RuleBased",
    "RandomForest",
    "XGBoost",
]:
    fused_path = (
        MAP_DIR
        / (
            f"Q1_FusedHybrid_{model_name}"
            "_Classification.tif"
        )
    )

    planet_path = (
        MAP_DIR
        / (
            f"Q1_PlanetOnly_{model_name}"
            "_Classification.tif"
        )
    )

    disagreement_path = (
        MAP_DIR
        / (
            f"Q1_{model_name}_Fused_vs_Planet"
            "_Disagreement.tif"
        )
    )

    with rasterio.open(
        fused_path
    ) as fused_src:
        with rasterio.open(
            planet_path
        ) as planet_src:
            profile = fused_src.profile.copy()
            profile.update(
                dtype="uint8",
                nodata=CLASS_NODATA,
                compress="DEFLATE",
                tiled=True,
                blockxsize=MAP_BLOCK_SIZE,
                blockysize=MAP_BLOCK_SIZE,
            )

            counts = {
                0: 0,
                1: 0,
                2: 0,
                3: 0,
            }

            with rasterio.open(
                disagreement_path,
                "w",
                **profile,
            ) as destination:
                for window in iter_windows(
                    fused_src.width,
                    fused_src.height,
                    MAP_BLOCK_SIZE,
                ):
                    fused_array = (
                        fused_src.read(
                            1,
                            window=window,
                        )
                    )

                    planet_array = (
                        planet_src.read(
                            1,
                            window=window,
                        )
                    )

                    valid = (
                        (fused_array != CLASS_NODATA)
                        & (
                            planet_array
                            != CLASS_NODATA
                        )
                    )

                    output = np.full(
                        fused_array.shape,
                        CLASS_NODATA,
                        dtype=np.uint8,
                    )

                    output[
                        valid
                        & (fused_array == 0)
                        & (planet_array == 0)
                    ] = 0

                    output[
                        valid
                        & (fused_array == 1)
                        & (planet_array == 1)
                    ] = 1

                    output[
                        valid
                        & (fused_array == 1)
                        & (planet_array == 0)
                    ] = 2

                    output[
                        valid
                        & (fused_array == 0)
                        & (planet_array == 1)
                    ] = 3

                    destination.write(
                        output,
                        1,
                        window=window,
                    )

                    for code in counts:
                        counts[code] += int(
                            np.sum(
                                output == code
                            )
                        )

            valid_total = sum(
                counts.values()
            )

            agreement_rows.append({
                "model": model_name,
                "both_nonboro_pixels": counts[0],
                "both_boro_pixels": counts[1],
                "fused_only_boro_pixels": counts[2],
                "planet_only_boro_pixels": counts[3],
                "valid_comparison_pixels": valid_total,
                "agreement_percent": (
                    100.0
                    * (
                        counts[0]
                        + counts[1]
                    )
                    / valid_total
                    if valid_total > 0
                    else np.nan
                ),
                "disagreement_path": str(
                    disagreement_path
                ),
            })

consensus_table = pd.DataFrame(
    consensus_rows
)

stream_map_agreement = pd.DataFrame(
    agreement_rows
)

consensus_table.to_csv(
    TABLE_DIR
    / "Q1_Consensus_Map_Inventory.csv",
    index=False,
)

stream_map_agreement.to_csv(
    TABLE_DIR
    / "Q1_Fused_vs_Planet_Map_Agreement.csv",
    index=False,
)

display(stream_map_agreement.round(4))

print("✅ Consensus and stream-disagreement maps created.")


In [ ]:
# CELL 19 — Final Excel workbook, manifest and Q1 recommendations

excel_path = (
    OUTPUT_ROOT
    / "Q1_Tanore_Classification_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:
    input_inventory.to_excel(
        writer,
        sheet_name="Input_Inventory",
        index=False,
    )

    sample_summary.to_excel(
        writer,
        sheet_name="Sample_Summary",
        index=False,
    )

    tuning_summary.to_excel(
        writer,
        sheet_name="Tuning_Thresholds",
        index=False,
    )

    performance_table.to_excel(
        writer,
        sheet_name="Validation_Performance",
        index=False,
    )

    pairwise_comparisons.to_excel(
        writer,
        sheet_name="Paired_Comparisons",
        index=False,
    )

    mapped_area_table.to_excel(
        writer,
        sheet_name="Mapped_Area",
        index=False,
    )

    stream_map_agreement.to_excel(
        writer,
        sheet_name="Stream_Map_Agreement",
        index=False,
    )

    phenology_table.to_excel(
        writer,
        sheet_name="NDVI_Phenology",
        index=False,
    )

    rf_importance_table.to_excel(
        writer,
        sheet_name="RF_Importance",
        index=False,
    )

    xgb_importance_table.to_excel(
        writer,
        sheet_name="XGB_Gain",
        index=False,
    )

    shap_importance_table.to_excel(
        writer,
        sheet_name="XGB_SHAP",
        index=False,
    )

    map_inventory.to_excel(
        writer,
        sheet_name="Map_Inventory",
        index=False,
    )


recommendations = f'''
TANORE BORO RICE — Q1 PUBLICATION RECOMMENDATIONS

1. Independent validation
Use a validation layer collected independently from training.
Target at least 150-200 spatially independent samples per class.
Do not split pixels from the same field between training and validation.

2. Sampling design
For defensible area-adjusted accuracy and area estimates, use a
probability-based stratified random validation design and retain
sampling inclusion probabilities.

3. Multi-year or external validation
A stronger Q1 paper should validate the workflow in another Boro season,
another upazila, or both. One place and one season limits generalizability.

4. Fusion ablation
Report:
- Planet-only classification
- Fused-Hybrid classification
- Rule-based baseline
- Random Forest
- XGBoost
The same validation samples must be used for all comparisons.

5. Temporal ablation
Add supplementary experiments removing one date at a time. This shows
which phenological stage contributes most to Boro discrimination.

6. Spatial leakage control
Use field-level groups for polygons and spatial blocks for points.
Pixel-random splitting should not be used for the final manuscript.

7. Statistical reporting
Report point estimates, cluster-bootstrap 95% confidence intervals,
McNemar exact p-values, paired F1 differences and practical effect sizes.
Do not claim superiority based only on a small numerical difference.

8. Explainability
Interpret RF permutation importance and XGBoost SHAP contributions in
relation to Boro phenology. Avoid treating importance as causal evidence.

9. Area reporting
The notebook reports mapped area. Call it mapped area, not unbiased area.
Use an Olofsson-style probability-sampling framework before claiming an
accuracy-adjusted crop area estimate.

10. Sensor and date mismatch
Discuss the 2-3 day Planet/Sentinel-2 acquisition gaps and the 7 April
Planet versus 10 April Sentinel-2 gap. Include this as an uncertainty
source.

11. Reproducibility
Archive:
- exact input filenames and dates
- CRS and resolution
- cloud/UDM2 masking rules
- fusion parameters
- random seed
- software versions
- trained models
- thresholds
- all independent validation predictions

12. Publication claims
The defensible central claim is:
phenology-aware multi-temporal fusion improves or does not improve Boro
rice discrimination relative to a Planet-only baseline under spatially
independent validation.
The result, not the expectation, must determine the conclusion.

VALIDATION DESIGN USED BY THIS NOTEBOOK:
{validation_design}
'''

recommendation_path = (
    REPORT_DIR
    / "Q1_Methodological_Recommendations.txt"
)

recommendation_path.write_text(
    recommendations,
    encoding="utf-8",
)

manifest = {
    "project": (
        "Phenology-based mapping of Boro rice "
        "in Tanore Upazila"
    ),
    "validation_design": validation_design,
    "streams": {
        stream_name: {
            data_type: {
                date_name: str(path)
                for date_name, path
                in values.items()
            }
            for data_type, values
            in stream.items()
        }
        for stream_name, stream
        in STREAMS.items()
    },
    "models": [
        "Data-calibrated phenology rule",
        "Random Forest",
        "XGBoost",
    ],
    "feature_names": FEATURE_NAMES,
    "random_seed": RANDOM_SEED,
    "spatial_block_size_m": (
        SPATIAL_BLOCK_SIZE_METERS
    ),
    "maximum_pixels_per_reference_feature": (
        MAX_PIXELS_PER_FEATURE
    ),
    "bootstrap_replicates": (
        BOOTSTRAP_REPLICATES
    ),
    "output_root": str(
        OUTPUT_ROOT
    ),
    "software": {
        "python": platform.python_version(),
        "rasterio": rasterio.__version__,
        "geopandas": gpd.__version__,
        "xgboost": xgb.__version__,
    },
}

manifest_path = (
    REPORT_DIR
    / "Q1_Classification_Reproducibility_Manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("✅ FINAL Q1 CLASSIFICATION WORKFLOW COMPLETE")
print()
print("Excel results:", excel_path)
print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)
print("GeoTIFF maps:", MAP_DIR)
print("Saved models:", MODEL_DIR)
print("Reports:", REPORT_DIR)
print("Recommendations:", recommendation_path)
print("Manifest:", manifest_path)
